In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error, r2_score

In [29]:
pip install openpyxl


[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
df = pd.read_excel('dataset/kerala_land_price_dataset.xlsx', sheet_name='Land_Data')

In [31]:
df.head()

,Year,District,Place,Land_Type,Road_Access,Distance_Town_km,Distance_Highway_m,Area_Cent,Land_Price_Per_Cent_INR,Source,Data_Status
0,2010,Thiruvananthapuram,Kazhakkoottam,NH 66 Bypass Forntage,YES,1.0,500,1,350000,NaN,Actual price required
1,2011,Thiruvananthapuram,Kazhakkoottam,NH 66 Bypass Forntage,YES,1.5,530,1,457000,NaN,Actual price required
2,2012,Thiruvananthapuram,Kazhakkoottam,NH 66 Bypass Forntage,YES,2.0,532,1,525000,NaN,Actual price required
3,2013,Thiruvananthapuram,Kazhakkoottam,NH 66 Bypass Forntage,YES,1.0,534,1,600000,NaN,Actual price required
4,2014,Thiruvananthapuram,Kazhakkoottam,NH 66 Bypass Forntage,YES,1.0,536,1,776000,NaN,Actual price required


In [32]:
df.shape

(119, 11)

## 3. Data Cleaning

In [33]:
df['Place'] = df['Place'].str.strip() #remove extra space
df['Land_Type'] = df['Land_Type'].str.strip()
df['Land_Type'] = df['Land_Type'].str.replace('Forntage', 'Frontage') #correct the spelling

In [34]:
df['Place'].unique()

array(['Kazhakkoottam', 'Akkulam', 'Vizhinjam', 'Pattom', 'Karyavattom',
       'Kavadior', 'East Fort'], dtype=object)

In [35]:
df['Land_Type'].unique()

array(['NH 66 Bypass Frontage', 'NH HIGHWAY Frontage',
       'Prime / Main Road / Commercial'], dtype=object)

##### Useless columns drop

In [36]:
df = df.drop(columns=['District', 'Road_Access', 'Area_Cent', 'Source', 'Data_Status'])

In [37]:
df = df.rename(columns={'Land_Price_Per_Cent_INR': 'Price'})

In [38]:
df.head()

,Year,Place,Land_Type,Distance_Town_km,Distance_Highway_m,Price
0,2010,Kazhakkoottam,NH 66 Bypass Frontage,1.0,500,350000
1,2011,Kazhakkoottam,NH 66 Bypass Frontage,1.5,530,457000
2,2012,Kazhakkoottam,NH 66 Bypass Frontage,2.0,532,525000
3,2013,Kazhakkoottam,NH 66 Bypass Frontage,1.0,534,600000
4,2014,Kazhakkoottam,NH 66 Bypass Frontage,1.0,536,776000


In [39]:
df = df.sort_values(['Place', 'Year']).reset_index(drop=True)

In [40]:
df.groupby('Place')['Year'].agg(['min', 'max', 'count'])

,min,max,count
Place,,,
Akkulam,2010,2026,17
East Fort,2010,2026,17
Karyavattom,2010,2026,17
Kavadior,2010,2026,17
Kazhakkoottam,2010,2026,17
Pattom,2010,2026,17
Vizhinjam,2010,2026,17


####  Feature Engineering

In [41]:
df['t'] = df['Year'] - 2010
df['log_price'] = np.log(df['Price'])
df['YoY_%'] = df.groupby('Place')['Price'].pct_change() * 100

In [42]:
df.head()

,Year,Place,Land_Type,Distance_Town_km,Distance_Highway_m,Price,t,log_price,YoY_%
0,2010,Akkulam,NH 66 Bypass Frontage,1.0,300,350000,0,12.765688,NaN
1,2011,Akkulam,NH 66 Bypass Frontage,1.0,320,420000,1,12.948010,20.000000
2,2012,Akkulam,NH 66 Bypass Frontage,1.0,300,550000,2,13.217674,30.952381
3,2013,Akkulam,NH 66 Bypass Frontage,1.0,300,600000,3,13.304685,9.090909
4,2014,Akkulam,NH 66 Bypass Frontage,1.0,300,750000,4,13.527828,25.000000


####  Outlier Check (IQR method)

In [43]:
q1 = df['YoY_%'].quantile(0.25)
q3 = df['YoY_%'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(lower, upper)

-9.166666666666682 37.5


In [44]:
df[(df['YoY_%'] < lower) | (df['YoY_%'] > upper)][['Place', 'Year', 'Price', 'YoY_%']]

,Place,Year,Price,YoY_%
18,East Fort,2011,1400000,40.000000
37,Karyavattom,2013,600000,50.000000
38,Karyavattom,2014,900000,50.000000
39,Karyavattom,2015,1500000,66.666667
53,Kavadior,2012,600000,50.000000


#### Save the Processed Data

In [45]:
df.to_csv('dataset/land_price_processed.csv', index=False)

In [46]:
df.shape

(119, 9)

####  Prepare for Modeling

In [47]:
pivot = df.pivot(index='Year', columns='Place', values='Price')

In [48]:
# Historical CAGR 2010-2026 (used later in the sanity check)
cagr = ((pivot.iloc[-1] / pivot.iloc[0]) ** (1 / 16) - 1) * 100

#### Train / Test Split (Time based)
- **Train:** 2010–2022
- **Test:** 2023–2026

In [49]:
train_df = df[df['Year'] < 2023]
test_df = df[df['Year'] >= 2023]

In [50]:
print(train_df.shape, test_df.shape)

(91, 9) (28, 9)


In [51]:
train = pivot[pivot.index < 2023]
test = pivot[pivot.index >= 2023]
places = pivot.columns

In [52]:
t_train = train.index.values - 2010
t_test = test.index.values - 2010
actual = test.values

#### Model Building

In [54]:
pred_naive = np.tile(train.values[-1], (len(test), 1))

#### Linear model

In [55]:
coef_lin = np.polyfit(t_train, train.values, 1)
pred_lin = np.clip(np.vander(t_test, 2) @ coef_lin, 0, None)

### Compound growth model

In [56]:
coef_comp = np.polyfit(t_train, np.log(train.values), 1)
pred_comp = np.exp(np.vander(t_test, 2) @ coef_comp)

#### Polynomial (degree 2) model

In [57]:
coef_poly = np.polyfit(t_train, train.values, 2)
pred_poly = np.clip(np.vander(t_test, 3) @ coef_poly, 0, None)

#### Tree models (Random Forest, Gradient Boosting)

In [58]:
X = pd.get_dummies(df[['Place', 'Year']], columns=['Place'])
X_train = X[df['Year'] < 2023]
X_test = X[df['Year'] >= 2023]
Y_train = df.loc[df['Year'] < 2023, 'Price']

In [59]:
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, Y_train)

,n_estimators,300
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [60]:
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, Y_train)

,loss,'squared_error'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [61]:
test_pred = test_df[['Place', 'Year']].copy()
test_pred['rf'] = rf.predict(X_test)
test_pred['gb'] = gb.predict(X_test)

In [62]:
pred_rf = test_pred.pivot(index='Year', columns='Place', values='rf').values
pred_gb = test_pred.pivot(index='Year', columns='Place', values='gb').values